# PDF Ingestion Pipeline

This notebook demonstrates the full pipeline for ingesting a PDF document into a vector database (ChromaDB) for RAG (Retrieval-Augmented Generation). The pipeline has four main steps:

1. **Parse** - Extract structured elements (text, headings, tables) from the PDF
2. **Chunk** - Split elements into smaller, semantically meaningful pieces
3. **Embed** - Convert text chunks into numerical vector representations
4. **Store** - Save vectors + metadata into ChromaDB for similarity search

In [1]:
# Auto-reload edited modules (e.g. rag/*) without restarting the kernel.
# Note: you must still recreate objects (like ChromaStore) after a class
# change for new methods to appear on existing instances.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Ensure the project root is in the path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.ingestion.pdf_parser import parse_pdf
from rag.ingestion.chunker import chunk_elements


## Step 1: Parse the PDF

`parse_pdf()` uses [Docling](https://github.com/DS4SD/docling) to convert a PDF into a list of `ParsedElement` objects. Each element has:
- **content** - the extracted text
- **type** - one of `"text"`, `"heading"`, or `"table"`
- **page** - the page number it came from
- **section_title** - the most recent heading (useful for context)

This preserves the document's structure rather than treating it as a flat string.

**Performance parameters** (Docling runs neural models on CPU, so parsing is the slowest step):
- **`do_table_structure`** - reconstruct table rows/columns with the TableFormer model. Keep `True` for financial filings so numbers stay aligned to their columns; `False` is faster but tables collapse to loose text.
- **`fast_tables`** - use TableFormer's `FAST` mode instead of `ACCURATE`. Big speedup on table-heavy docs with a small fidelity trade-off (default `True`).
- **`num_threads`** - CPU threads for inference (defaults to all cores).
- **`do_ocr`** - only needed for scanned/image PDFs; leave `False` for digital text (OCR is very slow).

The converter is cached internally, so re-running this cell reuses the already-loaded models.

In [2]:
elements = parse_pdf(
    project_root / "pdf/form-10-q.pdf", 
    do_table_structure=True, 
    fast_tables=True
)

In [3]:
for elem in elements[:5]:
    print(f"[Page {elem.page}] [{elem.type}] {elem.content[:80]}...")


[Page 1] [heading] UNITED STATES...
[Page 1] [heading] SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549...
[Page 1] [heading] FORM 10-Q...
[Page 1] [text] ☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE AC...
[Page 1] [text] For the quarterly period ended March 28, 2026...


In [4]:
elements[0:2]

[ParsedElement(content='UNITED STATES', type='heading', page=1, section_title='UNITED STATES', metadata={}),
 ParsedElement(content='SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549', type='heading', page=1, section_title='SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549', metadata={})]

## Step 2: Chunk the Elements

Raw parsed elements can be too long or too short for effective retrieval. Chunking splits them into pieces sized for the embedding model's context window while preserving semantic coherence.

Parameters:
- **method** - `"recursive"` splits by characters; `"semantic"` groups sentences by meaning similarity
- **chunk_size** - maximum characters per chunk (1000 is a good default for most embedding models)
- **chunk_overlap** - characters shared between adjacent chunks to avoid losing context at boundaries
- **keep_tables_intact** - prevents tables from being split mid-row

In [5]:
chunks = chunk_elements(
    elements,
    # method="recursive",
    method="semantic",
    chunk_size=1000,  # Max characters per chunk
    chunk_overlap=200,  # Characters of overlap between chunks
    keep_tables_intact=True,
)


In [6]:
print(f"Total chunks: {len(chunks)}")
print(
    f"First chunk ({len(chunks[0].content)} chars): {chunks[0].content[:100]}..."
)


Total chunks: 403
First chunk (907 chars): UNITED STATES

SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549

FORM 10-Q

☒ QUARTERLY REP...


In [7]:
print(chunks[0].content)


UNITED STATES

SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549

FORM 10-Q

☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the quarterly period ended March 28, 2026

or

☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the transition period from __________ to __________.

Commission File Number 001-38842

Delaware

83-0940635

State or Other Jurisdiction of Incorporation or Organization

I.R.S. Employer Identification

500 South Buena Vista Street Burbank, California 91521

Address of Principal Executive Offices and Zip Code

(818) 560-1000

Registrant's Telephone Number, Including Area Code

Securities registered pursuant to Section 12(b) of the Act:

Title of each class

Trading Symbol(s)

Name of each exchange on which registered

Common Stock, $0.01 par value

DIS

New York Stock Exchange


## Step 3: Generate Embeddings

Embeddings convert text into dense numerical vectors (arrays of floats) that capture semantic meaning. Similar texts produce vectors that are close together in vector space, enabling similarity search.

This step:
1. **Loads the config** - reads `config.yaml` for the active embedding provider (OpenAI, Gemini, etc.)
2. **Initializes the embedder** - creates a LangChain embedding client based on the config
3. **Embeds in batches** - sends chunks to the embedding API in batches of 50 to avoid rate limits and manage memory

In [8]:
from rag.config import load_config
from rag.ingestion.embedder import get_embedder
from rag.ingestion.store import ChromaStore
from rag.ingestion.embedder import embed_documents_batched

In [9]:
config = load_config()
print(config)

providers={'openai': ProviderConfig(api_key_env='OPENAI_API_KEY', models=ModelsConfig(embedding=ModelConfig(name='text-embedding-3-small', dimensions=1536, temperature=None, max_tokens=None), chat=ModelConfig(name='gpt-4o', dimensions=None, temperature=0.0, max_tokens=2048))), 'anthropic': ProviderConfig(api_key_env='ANTHROPIC_API_KEY', models=ModelsConfig(embedding=None, chat=ModelConfig(name='claude-sonnet-4-20250514', dimensions=None, temperature=0.0, max_tokens=2048))), 'gemini': ProviderConfig(api_key_env='GEMINI_API_KEY', models=ModelsConfig(embedding=ModelConfig(name='text-embedding-004', dimensions=768, temperature=None, max_tokens=None), chat=ModelConfig(name='gemini-2.5-flash', dimensions=None, temperature=0.0, max_tokens=2048)))} active=ActiveConfig(embedding_provider='openai', chat_provider='openai') chunking=ChunkingConfig(method='recursive', chunk_size=1000, chunk_overlap=200, keep_tables_intact=True) retrieval=RetrievalConfig(top_k=5, rerank=True, rerank_model='cross-enc

In [10]:
embedder = get_embedder(config)
print(embedder)


client=<openai.resources.embeddings.Embeddings object at 0xfff54cd04510> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0xfffdf5f38c90> model='text-embedding-3-small' dimensions=1536 deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True


In [11]:
vectors = embed_documents_batched(
    embedder, [c.content for c in chunks], batch_size=50
)


In [12]:
vectors[0][0:9]


[0.0531005859375,
 0.0220947265625,
 0.03515625,
 0.052032470703125,
 -0.0034198760986328125,
 0.01000213623046875,
 -0.007343292236328125,
 0.034576416015625,
 0.00531768798828125]

## Step 4: Store in ChromaDB

Now we have chunks (text) and vectors (embeddings) — the final step is storing them in ChromaDB so we can query them later. Each record in ChromaDB has four components:

| Component | Purpose |
|-----------|---------|
| **id** | A deterministic hash so re-ingesting the same document overwrites rather than duplicates |
| **document** | The raw chunk text, returned at query time so you don't need a separate lookup |
| **metadata** | Source file, page number, chunk type, section, and a human-friendly `label` — enables filtered queries |
| **embedding** | The vector used for similarity search |

**Metadata `label`** — a friendly document name (e.g. `"Disney Q2 2026 Earnings"`) stored on every chunk. Useful for display and for filtering queries to a specific document later.

**Idempotency & the `overwrite` guard** — IDs are deterministic (`source_file` + chunk index + content), so `upsert` overwrites matching records instead of appending. Re-running with the same document therefore keeps the chunk count stable rather than duplicating. The store cell below adds an explicit guard:
- If the document is **not** yet stored → it is ingested.
- If it **is** already stored and `overwrite=False` (default) → the ingestion is **skipped** with a message.
- If it is already stored and `overwrite=True` → existing chunks for that `source_file` are **deleted first**, then re-inserted (this also clears any now-stale chunks from a previous, differently-chunked version).

In [13]:
import hashlib
from datetime import datetime, timezone  

In [14]:
# 1. Connect to ChromaDB
store = ChromaStore(config)
store

In [15]:
# 2. Build IDs and metadata for each chunk
source_file = "form-10-q.pdf"
label = "Disney Q2 2026 Earnings"  # Human-friendly document label for filtering/display
timestamp = datetime.now(timezone.utc).isoformat()

In [16]:
ids = []
metadatas = []
documents = []

for i, chunk in enumerate(chunks):
    # Deterministic ID based on source + index + content
    hash_input = f"{source_file}:{i}:{chunk.content[:100]}"
    chunk_id = hashlib.sha256(hash_input.encode()).hexdigest()[:32]
    ids.append(chunk_id)

    documents.append(chunk.content)
    metadatas.append(
        {
            "source_file": source_file,
            "label": label,
            "page_number": chunk.page,
            "chunk_type": chunk.type,
            "section_title": chunk.section_title,
            "chunk_index": chunk.chunk_index,
            "ingestion_timestamp": timestamp,
        }
    )



In [17]:
# 3. Store into ChromaDB (pass your pre-computed embeddings)
#
# By default we refuse to re-ingest a document that is already stored,
# so repeated runs don't silently rewrite data. Set overwrite=True to
# replace it: existing chunks for this source_file are deleted first,
# then the new ones are inserted (this also clears any now-stale chunks
# from a previous, differently-chunked version of the same file).
overwrite = False

if store.document_exists(source_file):
    existing = store.count_by_source(source_file)
    if overwrite:
        print(f"'{source_file}' already stored ({existing} chunks) — overwriting.")
        store.delete_by_source(source_file)
        store.upsert(
            ids=ids,
            documents=documents,
            metadatas=metadatas,
            embeddings=vectors,
        )
        print(f"Re-ingested {len(ids)} chunks.")
    else:
        print(
            f"Skipped: '{source_file}' is already stored ({existing} chunks). "
            f"Set overwrite=True to replace it."
        )
else:
    store.upsert(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=vectors,
    )
    print(f"Ingested {len(ids)} chunks.")

Skipped: 'form-10-q.pdf' is already stored (403 chunks). Set overwrite=True to replace it.


In [18]:
# 4. Verify
print(f"Stored {store.count()} chunks in collection")


Stored 403 chunks in collection
